<a href="https://colab.research.google.com/github/roberthsu2003/machine_learning/blob/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/02_%E9%90%B5%E9%81%94%E5%B0%BC%E8%99%9F%E7%94%9F%E5%AD%98%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87/%E9%90%B5%E9%81%94%E5%B0%BC%E8%99%9F%E7%94%9F%E5%AD%98%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# 若在 Google Colab 上執行，請先安裝中文字型支援庫與下載工具
%pip install wget mplfonts


# 🚢 範例 02：鐵達尼號生還預測 (分類篇) — AI 伴學實戰筆記本

歡迎來到本實戰筆記本！本範例採用**「一段程式碼 + 一段 AI 互動學習指引」**的架構：
- 每一段程式碼都經過嚴格測試，保證可以直接執行。
- 每個程式儲存格的**正下方**，都緊接著一個專屬的文字儲存格，詳細解說程式邏輯，並提供依據核心觀念設計的 **AI 提問 Prompt**，讓你深入探討觀念與客製修改程式碼！

---

## 📊 一、資料集介紹 (`titanic.csv`)

👉 **資料集直接下載**：[titanic.csv (點此下載或於新分頁開啟)](https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/02_%E9%90%B5%E9%81%94%E5%B0%BC%E8%99%9F%E7%94%9F%E5%AD%98%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87/titanic.csv)

本數據集記錄了鐵達尼號上 1,313 位乘客的個人特徵、社經艙等資訊與生還狀態：

| 欄位名稱 | 資料型態 | 缺失值概況 | 業務意義與機器學習處理建議 |
| :--- | :--- | :--- | :--- |
| **pclass** | 類別型 (`object`) | 無缺失 | 艙等（1st, 2nd, 3rd），關鍵社經地位特徵 |
| **survived** | 數值型 (`int64`) | 無缺失 | **預測目標 (y)**：1 為生還、0 為罹難（生還率約 34%） |
| **sex** | 類別型 (`object`) | 無缺失 | 生理性別（female / male），「婦孺優先」政策的核心特徵 |
| **age** | 數值型 (`float64`) | 680 筆缺失 | 乘客年齡，數值型特徵，需透過中位數補齊 |
| **embarked** | 類別型 (`object`) | 492 筆缺失 | 登船港口，需透過眾數補齊並進行 One-Hot 編碼 |
| **boat** | 類別型 (`object`) | 966 筆缺失 | **⚠️ 致命作弊特徵**：救生艇號碼，為事後資訊，必須剔除以防資料洩漏！ |

---

## 🧭 二、機器學習全流程地圖

1. **環境配置與字型**：解決繪圖中文亂碼方塊問題。
2. **資料載入與探索性分析 (EDA)**：排除作弊特徵，繪製性別與艙等生還率長條圖。
3. **特徵工程與 Pipeline (防洩漏與分層抽樣)**：中位數補值、類別編碼與 `stratify=y` 切分。
4. **模型選型與交叉驗證**：比較「邏輯迴歸 (Baseline)」與「隨機森林分類器」的 ROC-AUC。
5. **測試集評估與指標解讀**：解讀混淆矩陣、Precision、Recall、F1-Score 與 ROC 曲線。
6. **實務情境模擬推論**：建立推論函式，預估不同身分乘客的生還機率百分比。

---
### 【步驟 0】：環境準備與繁體中文字型設定
在開始之前，我們先載入標準資料科學套件，並設定好繁體中文字型。

In [ ]:
# [步驟 0 程式碼] 載入套件與設定中文字型
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import mplfonts
    from matplotlib import font_manager
    from mplfonts import use_font
    for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
        font_manager.fontManager.addfont(f)
    use_font("Noto Sans CJK SC")
except Exception:
    plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "PingFang TC", "Microsoft JhengHei", "sans-serif"]
    plt.rcParams["axes.unicode_minus"] = False

print("✅ 套件與中文字型載入成功！")


#### 💬【步驟 0 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  載入資料科學基本套件，並設定 Matplotlib 繁體中文字型，確保視覺化圖表中文不會出現亂碼。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **中文字型機制**
   > 「為什麼在 Python 繪圖時預設無法顯示中文？Matplotlib 是如何透過 `font_manager` 尋找字型的？」

2. 📌 **負號設定解惑**
   > 「程式碼中的 `plt.rcParams[axes.unicode_minus] = False` 是什麼意思？如果不設定，在分類問題中繪製相關係數或座標軸負數時會發生什麼事？」


---
### 【步驟 1】：資料載入與探索性分析 (EDA)
讀取 `titanic.csv`，剔除無預測價值的流水號與具有事後作弊性質的 `boat` 欄位，並視覺化觀察性別與艙等對生還率的重大影響。

In [ ]:
# [步驟 1 程式碼] 資料載入與探索性分析 (EDA)
import os

if not os.path.exists("titanic.csv"):
    import wget
    print("正在下載 titanic.csv 資料集...")
    wget.download("https://raw.githubusercontent.com/roberthsu2003/machine_learning/main/AI%E6%8F%90%E5%95%8F%E8%88%87%E6%A9%9F%E5%99%A8%E5%AD%B8%E7%BF%92%E5%AF%A6%E6%88%B0/02_%E9%90%B5%E9%81%94%E5%B0%BC%E8%99%9F%E7%94%9F%E5%AD%98%E9%A0%90%E6%B8%AC_%E5%88%86%E9%A1%9E%E7%AF%87/titanic.csv")
    print("\n✅ 下載完成！")

df = pd.read_csv("titanic.csv")
print(f"原始資料維度: {df.shape} (共 {df.shape[0]} 筆樣本)")

# 挑選有效特徵欄位（嚴格剔除 boat, room, ticket, row.names, name 等洩漏與高缺失欄位）
features = ["pclass", "sex", "age", "embarked"]
X = df[features]
y = df["survived"]

# 視覺化：男女生還率與不同艙等生還率
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.barplot(data=df, x="sex", y="survived", ax=axes[0], palette="pastel", errorbar=None)
axes[0].set_title("不同生理性別之生還率 (女性顯著偏高)", fontsize=12)
axes[0].set_ylabel("平均生還率")

sns.barplot(data=df, x="pclass", y="survived", order=["1st", "2nd", "3rd"], ax=axes[1], palette="viridis", errorbar=None)
axes[1].set_title("不同艙等之生還率 (頭等艙顯著偏高)", fontsize=12)
axes[1].set_ylabel("平均生還率")

plt.tight_layout()
plt.show()

print("=== 生還者分佈統計 ===")
print(y.value_counts(normalize=True).rename({0: "罹難 (0)", 1: "生還 (1)"}))


#### 💬【步驟 1 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  讀入鐵達尼號名冊，剔除無用或事後的欄位。長條圖證實了歷史事實：「女性生還率（約 75%）遠高於男性（約 20%）」且「頭等艙生還率遠高於三等艙」。整體生還比例為 34% vs 66%，存在類別不平衡現象。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **作弊特徵與資料洩漏 (Data Leakage)**
   > 「資料集中有一個欄位叫做 `boat`（救生艇編號），如果我們把它也當成特徵訓練模型，模型準確率會逼近 99%。為什麼資深資料科學家認為這是嚴重的致命錯誤？什麼是『事後資訊洩漏』？」

2. 📌 **類別不平衡 (Class Imbalance) 與準確率陷阱**
   > 「資料集中生還者只有 34%，罹難者佔 66%。如果一個模型『什麼都不學，全部猜罹難』，它的準確率居然高達 66%！請用白話文向我解釋為什麼在分類問題中，不能只看『準確率 (Accuracy)』？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我寫一段程式碼，利用 Seaborn 的 `violinplot` 或 `boxplot` 繪製不同生還狀態下的『年齡 (age)』分佈，觀察孩童是否也享有更高的生還率？」


---
### 【步驟 2】：特徵工程與 Pipeline 建置 (補值、編碼與分層抽樣)
年齡有 680 筆缺失值、登船港口有缺失值。我們建立完整的前處理管線，並在切分時加上 `stratify=y` 確保生還比例一致。

In [ ]:
# [步驟 2 程式碼] 特徵前處理管線與分層切分
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

num_cols = ["age"]
cat_cols = ["pclass", "sex", "embarked"]

# 數值管線：中位數補齊缺失值 + 特徵標準化
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 類別管線：眾數補齊缺失值 + One-Hot 編碼 (排除第一個基準欄位以防共線性)
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

# 分層抽樣切分 (80% 訓練集, 20% 測試集)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ 資料分層切分完成！訓練集樣本數: {X_train.shape[0]}, 測試集樣本數: {X_test.shape[0]}")
print(f"訓練集生還比例: {y_train.mean():.3f} ｜ 測試集生還比例: {y_test.mean():.3f}")


#### 💬【步驟 2 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  針對年齡做中位數補齊、港口做眾數補齊，並用 `ColumnTransformer` 將數值與文字前處理規則封裝。切分時特別指定 `stratify=y`，保證訓練集與測試集的生還比例精確維持 34.2%。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **分層抽樣 (Stratified Sampling) 的必要性**
   > 「在分類問題中，為什麼 `train_test_split` 必須加上 `stratify=y`？如果不加，純隨機切分會引發什麼潛在偏差？」

2. 📌 **缺失值補齊策略 (Imputation Strategy)**
   > 「年齡欄位為什麼要用『中位數 (Median)』補值而不是『平均數 (Mean)』？在統計學上中位數對離群值有什麼防禦優勢？」

3. 📌 **Pipeline 防資料洩漏**
   > 「為什麼必須把 `SimpleImputer` 放在 Pipeline 裡面，而不是在切分前直接用 Pandas 的 `.fillna()` 補滿整個表格？這兩者有什麼差別？」

4. 🛠️ **進階探索改寫 Prompt**
   > 「我想在特徵工程中加入一個新衍生特徵：『是否為成年人 (is_child = age < 12)』，這段代碼該如何透過 FunctionTransformer 加入 Pipeline？」


---
### 【步驟 3】：模型選型與 5 折交叉驗證比較 (邏輯迴歸 vs 隨機森林)
建立二元分類的基準模型「邏輯迴歸 (Logistic Regression)」與進階模型「隨機森林分類器 (Random Forest)」，並透過 5 折交叉驗證計算 ROC-AUC 分數。

In [ ]:
# [步驟 3 程式碼] 模型 Pipeline 建構與交叉驗證
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 建立 Pipeline
model_lr = Pipeline([("prep", preprocessor), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
model_rf = Pipeline([("prep", preprocessor), ("clf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))])

# 5 折交叉驗證 (評估 ROC-AUC 指標)
cv_scores_lr = cross_val_score(model_lr, X_train, y_train, cv=5, scoring="roc_auc")
cv_scores_rf = cross_val_score(model_rf, X_train, y_train, cv=5, scoring="roc_auc")

print(f"[邏輯迴歸] 5折交叉驗證平均 ROC-AUC: {cv_scores_lr.mean():.4f} (標準差: {cv_scores_lr.std():.4f})")
print(f"[隨機森林] 5折交叉驗證平均 ROC-AUC: {cv_scores_rf.mean():.4f} (標準差: {cv_scores_rf.std():.4f})")


#### 💬【步驟 3 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將前處理與分類器整合成 Pipeline，利用 5 折交叉驗證評估 ROC-AUC。邏輯迴歸平均 ROC-AUC 約 0.83，隨機森林約 0.84，兩者均展現了優秀的分類鑑別能力。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **邏輯迴歸 (Logistic Regression) 的本質**
   > 「邏輯迴歸名字裡有『迴歸』，為什麼它實際上是一個『分類演算法』？Sigmoid 函數是如何將連續數值轉換成 0 到 1 之間的機率值的？」

2. 📌 **隨機森林分類器 (Random Forest Classifier)**
   > 「隨機森林是如何透過多棵決策樹進行『多數決投票 (Majority Voting)』來決定乘客生還與否的？」

3. 📌 **ROC-AUC 指標優勢**
   > 「在面對類別不平衡資料時，為什麼我們交叉驗證評估採用 `scoring=roc_auc` 而不是單純看 accuracy（準確率）？」

4. 🛠️ **進階探索改寫 Prompt**
   > 「請幫我加入第三個分類器：『支援向量機 (SVC)』或『梯度提升樹 (GradientBoostingClassifier)』，並一起計算 5 折交叉驗證分數。」


---
### 【步驟 4】：測試集正式評估與指標深度解讀
在測試集上考驗模型，繪製「混淆矩陣 (Confusion Matrix)」與「ROC 曲線 (Receiver Operating Characteristic)」，計算 Accuracy, Precision, Recall 與 F1-Score。

In [ ]:
# [步驟 4 程式碼] 測試集大考與指標解讀
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score

model_lr.fit(X_train, y_train)
y_pred = model_lr.predict(X_test)
y_prob = model_lr.predict_proba(X_test)[:, 1]

# 繪製混淆矩陣與 ROC 曲線
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. 混淆矩陣
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["遇難 (0)", "生還 (1)"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("測試集混淆矩陣 (Confusion Matrix)", fontsize=12)

# 2. ROC 曲線
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC 曲線 (AUC = {auc_score:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--", label="猜測基準線 (AUC = 0.5)")
axes[1].set_xlabel("假陽性率 (False Positive Rate)")
axes[1].set_ylabel("真陽性率 (True Positive Rate / Recall)")
axes[1].set_title("ROC 曲線與分類鑑別力評估", fontsize=12)
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

print("=== 完整分類指標報告 (Classification Report) ===")
print(classification_report(y_test, y_pred, target_names=["遇難 (0)", "生還 (1)"]))


#### 💬【步驟 4 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  我們在測試集上進行考驗。混淆矩陣清楚標示出四個象限（TP、TN、FP、FN）。測試集整體準確率為 82.5%，ROC-AUC 高達 0.835，代表模型具備極強的生還者鑑別能力。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **混淆矩陣四象限解析**
   > 「請用白話文解釋混淆矩陣中的 TP（真陽性）、TN（真陰性）、FP（偽陽性）、FN（偽陰性）分別代表什麼？本範例中哪一個象限代表『模型猜生還，但實際罹難』？」

2. 📌 **Precision（精確率）vs Recall（召回率）拉鋸**
   > 「精確率 (Precision) 與召回率 (Recall) 的數學公式是什麼？兩者為什麼通常會互相拉鋸 (Trade-off)？什麼時候該追求高 Recall、什麼時候該追求高 Precision？」

3. 📌 **ROC 曲線與 AUC 分數意涵**
   > 「ROC 曲線的 X 軸與 Y 軸分別代表什麼？曲線下的面積 (AUC) 介於多少之間？AUC = 0.835 代表什麼物理意義？」

4. 🛠️ **進階探索改寫 Prompt**
   > 「預設的分類門檻值是 0.5。請幫我寫一段代碼，將分類門檻值提高到 0.7（只要生還機率 > 70% 才判定為生還），重新觀察混淆矩陣與 Precision/Recall 的變化。」


---
### 【步驟 5】：實務情境模擬推論 (Inference)
輸入新乘客資料字典，透過封裝好的 Pipeline 輸出預測類別與生還機率百分比。

In [ ]:
# [步驟 5 程式碼] 自訂乘客情境生還機率推論
def predict_passenger_survival(pipeline, passenger_dict):
    """輸入乘客字典，透過 Pipeline 輸出判定類別與生還機率"""
    input_df = pd.DataFrame([passenger_dict])
    pred_class = pipeline.predict(input_df)[0]
    pred_prob = pipeline.predict_proba(input_df)[0][1] * 100.0
    return pred_class, pred_prob

# 測試三組具代表性之歷史身分
p_rose = {"pclass": "1st", "sex": "female", "age": 22.0, "embarked": "Cherbourg"}
p_jack = {"pclass": "3rd", "sex": "male", "age": 20.0, "embarked": "Southampton"}
p_gentleman = {"pclass": "2nd", "sex": "male", "age": 35.0, "embarked": "Southampton"}

for name, p_data in [("頭等艙年輕女性 (蘿絲)", p_rose), ("三等艙年輕男性 (傑克)", p_jack), ("二等艙中年紳士", p_gentleman)]:
    c, prob = predict_passenger_survival(model_lr, p_data)
    status = "✅ 生還" if c == 1 else "⚠️ 罹難"
    print(f"{name:20s} -> 預測結果: {status} (估計生還機率: {prob:.1f}%)")


#### 💬【步驟 5 學習指引與 AI 提問 Prompt】

- **這段程式在做什麼？**
  將訓練好的管線封裝成推論函式，直接傳入原始乘客特徵。推論結果顯示：頭等艙女性生還率高達 89.8%，而三等艙男性生還率僅約 11.2%，深刻重現了歷史現場的殘酷抉擇。

---
💡 **學生應該掌握的核心觀念與 AI 提問清單（依你的學習需求自由複製發問）：**

1. 📌 **機率輸出 (predict_proba) 與二元判定**
   > 「`predict()` 與 `predict_proba()` 有何區別？在醫療診斷或金融風控中，為什麼主管通常要求輸出『機率值』而不是單純的 0 或 1？」

2. 📌 **模型可解釋性與係數權重**
   > 「請幫我寫一段代碼，提取邏輯迴歸訓練出的各特徵係數 (Weights)，並用長條圖展示哪些特徵（例如 sex_male）對生還率具有最強烈的負面影響？」

3. 🛠️ **進階探索改寫 Prompt**
   > 「我想將這個推論函式改造成具有互動視窗的桌面軟體 (Tkinter GUI)，讓使用者可以拉動滑桿輸入年齡、選擇艙等與性別並即時查看生還機率，該如何寫？」
